Варіант 17.

MLP-класифікація та порівняння з SVM.

In [1]:
from pathlib import Path
import warnings

import h5py
import numpy as np
import pandas as pd
from sklearn.exceptions import ConvergenceWarning
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC

warnings.filterwarnings("ignore", category=ConvergenceWarning)

Завантаження та нормалізація

In [2]:
data_dir = Path("data")
if not data_dir.exists():
    data_dir = Path("..") / "data"


def load_dataset(data_dir):
    with h5py.File(data_dir / "train_catvnoncat.h5", "r") as train_file:
        x_train = np.array(train_file["train_set_x"][:])
        y_train = np.array(train_file["train_set_y"][:]).ravel()
        classes = [item.decode("utf-8") for item in train_file["list_classes"][:]]

    with h5py.File(data_dir / "test_catvnoncat.h5", "r") as test_file:
        x_test = np.array(test_file["test_set_x"][:])
        y_test = np.array(test_file["test_set_y"][:]).ravel()

    return x_train, y_train, x_test, y_test, classes


x_train_img, y_train, x_test_img, y_test, classes = load_dataset(data_dir)

x_train = x_train_img.reshape(x_train_img.shape[0], -1) / 255.0
x_test = x_test_img.reshape(x_test_img.shape[0], -1) / 255.0

print("x_train:", x_train.shape)
print("x_test: ", x_test.shape)
print("classes:", classes)

x_train: (209, 12288)
x_test:  (50, 12288)
classes: ['non-cat', 'cat']


Навчання

In [3]:
experiments = [
    {"architecture": "Layer-1 [100]", "hidden_layer_sizes": (100,), "solver": "lbfgs", "activation": "identity", "max_iter": 200, "alpha": 0.0001},
    {"architecture": "Layer-2 [3, 3]", "hidden_layer_sizes": (3, 3), "solver": "lbfgs", "activation": "logistic", "max_iter": 200, "alpha": 0.0001},
    {"architecture": "Layer-2 [3, 3]", "hidden_layer_sizes": (3, 3), "solver": "lbfgs", "activation": "tanh", "max_iter": 200, "alpha": 0.0001},
    {"architecture": "Layer-3 [20, 7, 10]", "hidden_layer_sizes": (20, 7, 10), "solver": "lbfgs", "activation": "logistic", "max_iter": 200, "alpha": 0.0001},
    {"architecture": "Layer-3 [20, 7, 10]", "hidden_layer_sizes": (20, 7, 10), "solver": "lbfgs", "activation": "tanh", "max_iter": 200, "alpha": 0.0001},
    {"architecture": "Layer-3 [20, 7, 10]", "hidden_layer_sizes": (20, 7, 10), "solver": "lbfgs", "activation": "relu", "max_iter": 200, "alpha": 0.0001},
]

In [4]:
rows = []
mlp_models = {}

for params in experiments:
    model = MLPClassifier(
        hidden_layer_sizes=params["hidden_layer_sizes"],
        solver=params["solver"],
        activation=params["activation"],
        max_iter=params["max_iter"],
        alpha=params["alpha"],
        random_state=42,
    )
    model.fit(x_train, y_train)

    row = {
        "architecture": params["architecture"],
        "solver": params["solver"],
        "activation": params["activation"],
        "max_iter": params["max_iter"],
        "alpha": params["alpha"],
        "train_accuracy": model.score(x_train, y_train),
        "test_accuracy": model.score(x_test, y_test),
    }
    rows.append(row)
    mlp_models[(params["architecture"], params["activation"])] = model

mlp_results = pd.DataFrame(rows)
mlp_results.sort_values(["test_accuracy", "train_accuracy"], ascending=False)

,architecture,solver,activation,max_iter,alpha,train_accuracy,test_accuracy
4,"Layer-3 [20, 7, 10]",lbfgs,tanh,200,0.0001,0.995215,0.84
2,"Layer-2 [3, 3]",lbfgs,tanh,200,0.0001,0.846890,0.74
5,"Layer-3 [20, 7, 10]",lbfgs,relu,200,0.0001,1.000000,0.66
0,Layer-1 [100],lbfgs,identity,200,0.0001,1.000000,0.62
3,"Layer-3 [20, 7, 10]",lbfgs,logistic,200,0.0001,0.942584,0.56
1,"Layer-2 [3, 3]",lbfgs,logistic,200,0.0001,0.808612,0.56


In [5]:
best_mlp = (
    mlp_results.loc[mlp_results.groupby("architecture")["test_accuracy"].idxmax()]
    .sort_values("test_accuracy", ascending=False)
    .reset_index(drop=True)
)

best_mlp

,architecture,solver,activation,max_iter,alpha,train_accuracy,test_accuracy
0,"Layer-3 [20, 7, 10]",lbfgs,tanh,200,0.0001,0.995215,0.84
1,"Layer-2 [3, 3]",lbfgs,tanh,200,0.0001,0.846890,0.74
2,Layer-1 [100],lbfgs,identity,200,0.0001,1.000000,0.62


Порівняння з SVM

In [6]:
svm_settings = [
    {"kernel": "linear", "C": 1},
    {"kernel": "rbf", "C": 10, "gamma": 0.001},
]

svm_rows = []
for params in svm_settings:
    model = SVC(**params)
    model.fit(x_train, y_train)
    svm_rows.append({
        "model": "SVM",
        "architecture": params["kernel"],
        "train_accuracy": model.score(x_train, y_train),
        "test_accuracy": model.score(x_test, y_test),
    })

svm_results = pd.DataFrame(svm_rows)
svm_results

,model,architecture,train_accuracy,test_accuracy
0,SVM,linear,1.0,0.72
1,SVM,rbf,1.0,0.82


,model,architecture,train_accuracy,test_accuracy
0,SVM,linear,1.0,0.72
1,SVM,rbf,1.0,0.82


In [7]:
comparison = pd.concat(
    [
        best_mlp.assign(model="MLP")[["model", "architecture", "train_accuracy", "test_accuracy"]],
        svm_results,
    ],
    ignore_index=True,
).sort_values("test_accuracy", ascending=False).reset_index(drop=True)

comparison

,model,architecture,train_accuracy,test_accuracy
0,MLP,"Layer-3 [20, 7, 10]",0.995215,0.84
1,SVM,rbf,1.000000,0.82
2,MLP,"Layer-2 [3, 3]",0.846890,0.74
3,SVM,linear,1.000000,0.72
4,MLP,Layer-1 [100],1.000000,0.62


,model,architecture,train_accuracy,test_accuracy
0,MLP,"Layer-3 [20, 7, 10]",0.995215,0.84
1,SVM,rbf,1.000000,0.82
2,MLP,"Layer-2 [3, 3]",0.846890,0.74
3,SVM,linear,1.000000,0.72
4,MLP,Layer-1 [100],1.000000,0.62


Результати

In [8]:
winner = comparison.iloc[0]
print(
    f"Найкращий результат має {winner['model']} ({winner['architecture']}): "
    f"test accuracy = {winner['test_accuracy']:.2%}."
)

Найкращий результат має MLP (Layer-3 [20, 7, 10]): test accuracy = 84.00%.
Найкращий результат має MLP (Layer-3 [20, 7, 10]): test accuracy = 84.00%.
